<img src="./Imagenes/logo_UTN.svg" align="right" width="150" /> 

#### Teoría de los Circuitos II

# Trabajo semanal 2
#### Lucas Gallero
----

# Enunciado:

Se pide lo siguiente:

1. Establezca la función transferencia pasa bajos $T_{LPF}(s)$ mediante aproximación de máxima planicidad, por caso, Butterworth, de orden 6.

2. Obtenga respuesta de módulo, fase y diagrama de polos y ceros de forma cualitativa.

3. Implemente una red normalizada que responda a $T_{LPF}(s)$ mediante secciones de segundo orden (SOS) separadas por buffers.  
   En orden creciente de Q (Q1 < Q2 < Q3):

   - SOS_1: Sallen-Key
   - SOS_2: KHN
   - SOS_3: MFB

4. Indique las normas de frecuencia $\Omega_{\omega}$ e impedancia $\Omega_z$ adoptadas y los ajustes necesarios para obtener una ganancia de 10 dB en la banda de paso.

5. Verifique los resultados obtenidos analíticamente mediante simulaciones numéricas y/o simbólicas en Python y circuitales en LTSpice.

---

**Bonus:**

- +1 🧑‍🔬 Se pide implementar una función transferencia Butter pero de orden 7. ¿Puede reutilizar las SOSs obtenidas anteriormente? ¿Bastaría con cascadear una sección de primer orden?
- +1 🧑‍🔬 Indique qué modificaciones debería realizar y obtenga la red normalizada correspondiente.
- +1 💎 Verifique los resultados mediante simulaciones en Python y LTSpice.
- +1 🎓 Presentación en Jupyter Notebook.


In [36]:
# PyTC2: La librería para TC2
from pytc2.sistemas_lineales import pzmap, GroupDelay, bodePlot

from scipy.signal import TransferFunction
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import sympy as sp
from sympy.abc import s
from scipy import signal
# las librerías que usarremos las cargamos solo una vez.


# Resolución


## 1) Transferencia de Filtro Pasa Bajos Butterwoorth de orden 6

### <ins>Desarrollo de la respuesta Butterworth

Para analizar la respuesta en frecuencia de un filtro, se parte de la función de transferencia evaluada en el eje imaginario, es decir:

$$
T(j\omega)
$$

Como esta función es compleja, puede escribirse separando su parte real e imaginaria:

$$
T(j\omega) = \text{Re}[T(j\omega)] + j\,\text{Im}[T(j\omega)]
$$

Si se reemplaza $j\omega$  por $-j\omega$, la parte real permanece igual, mientras que la parte imaginaria cambia de signo:

$$
T(-j\omega) = \text{Re}[T(j\omega)] - j\,\text{Im}[T(j\omega)]
$$

Por lo tanto, $T(-j\omega)$ corresponde al conjugado complejo de $T(j\omega)$:

$$
T(-j\omega) = T^*(j\omega)
$$

La magnitud al cuadrado de una función compleja se obtiene multiplicando dicha función por su conjugada:

$$
|T(j\omega)|^2 = T(j\omega) \cdot T^*(j\omega)
$$

Como $T^*(j\omega) = T(-j\omega)$, se obtiene:

$$
|T(j\omega)|^2 = T(j\omega) \cdot T(-j\omega)
$$

Esta relación permite expresar la magnitud al cuadrado de la función de transferencia a partir de la función evaluada en $j\omega$ y en $-j\omega$.

Luego se introduce una función característica $K(j\omega)$, que representa la desviación de la respuesta respecto del comportamiento ideal de un filtro pasa bajos. La magnitud al cuadrado se escribe como:

$$
|T_n(j\omega)|^2 = \frac{1}{1 + |K(j\omega)|^2}
$$

En esta expresión, cuando $K(j\omega)$ es pequeño, el denominador tiende a 1 y, por lo tanto, la magnitud del filtro se aproxima a la unidad. Esto ocurre en la banda de paso. En cambio, cuando $K(j\omega)$ aumenta, el denominador crece y la magnitud disminuye, produciendo la atenuación propia de la banda de rechazo.

Para obtener una respuesta Butterworth, se impone que la magnitud sea máximamente plana alrededor de $\omega = 0$. Esto significa que la respuesta no debe presentar ondulaciones en la banda de paso y que sus primeras variaciones deben ser nulas.

Se considera entonces que la magnitud al cuadrado de la función característica puede escribirse como un polinomio par de la frecuencia:

$$
|K(j\omega)|^2 = B_2\omega^2 + B_4\omega^4 + B_6\omega^6 + \cdots + B_{2n}\omega^{2n}
$$

Para que la respuesta sea máximamente plana, se anulan todos los coeficientes de menor orden:

$$
B_2 = B_4 = B_6 = \cdots = B_{2(n-1)} = 0
$$

De esta manera, solamente queda el término de mayor orden:

$$
|K(j\omega)|^2 = B_{2n}\omega^{2n}
$$

Definiendo:

$$
B_{2n} = \varepsilon^2
$$

se obtiene:

$$
|K(j\omega)|^2 = \varepsilon^2 \omega^{2n}
$$

Finalmente, reemplazando esta expresión en la ecuación general de la magnitud:

$$
|T_n(j\omega)|^2 = \frac{1}{1 + |K(j\omega)|^2}
$$

resulta:

$$
|T_n(j\omega)|^2 = \frac{1}{1 + \varepsilon^2 \omega^{2n}}
$$

Esta expresión corresponde a la respuesta en magnitud al cuadrado de un filtro Butterworth de orden $n$. Para el caso normalizado, se toma $\varepsilon^2 = 1$, por lo que queda:

$$
\boxed{|T_n(j\omega)|^2 = \frac{1}{1 + \omega^{2n}}}
$$

Esta es la forma clásica de la respuesta Butterworth normalizada, cuya principal característica es presentar una banda de paso máximamente plana y una pendiente de atenuación que aumenta con el orden del filtro.

### <ins>Obtención de la función transferencia Butterworth de orden 6

En este caso, el enunciado solicita obtener la función transferencia pasa bajos mediante una aproximación de máxima planicidad, es decir, una aproximación Butterworth de orden 6. Por lo tanto:

$$
n = 6
$$

Para un filtro Butterworth normalizado, la respuesta en magnitud al cuadrado está dada por:

$$
|T_n(j\omega)|^2 = \frac{1}{1+\omega^{2n}}
$$

Reemplazando $n = 6$:

$$
|T_6(j\omega)|^2 = \frac{1}{1+\omega^{12}}
$$

Esta expresión describe únicamente la magnitud al cuadrado del filtro. Para obtener la función transferencia en el dominio de Laplace, se utiliza la siguiente relación:

$$
|T(j\omega)|^2 = T(s)T(-s)
$$

Por lo tanto, para el caso Butterworth de orden 6:

$$
T_6(s)T_6(-s) = \frac{1}{1+\omega^{12}}
$$

Como se trabaja sobre el eje imaginario, se cumple que:

$$
s = j\omega
$$

de donde:

$$
\omega = \frac{s}{j}
$$

Entonces:

$$
\omega^{12} = \left(\frac{s}{j}\right)^{12}
$$

Como:

$$
j^{12} = 1
$$

resulta:

$$
\omega^{12} = s^{12}
$$

Por lo tanto:

$$
T_6(s)T_6(-s) = \frac{1}{1+s^{12}}
$$

Para encontrar los polos de la transferencia, se iguala a cero el denominador:

$$
1+s^{12}=0
$$

Luego:

$$
s^{12} = -1
$$

Esta ecuación posee doce raíces distribuidas sobre una circunferencia de radio unitario en el plano complejo. Sin embargo, para construir una función transferencia estable, se seleccionan únicamente las raíces ubicadas en el semiplano izquierdo, es decir, aquellas con parte real negativa.

Los ángulos absolutos de los polos estables, medidos desde el eje real positivo, pueden calcularse mediante:

$$
\boxed{\theta_k = \frac{\pi}{2} + \frac{(2k-1)\pi}{2n}}
$$

donde:

$$
k = 1,2,\dots,n
$$

Como en este caso $n = 6$:

$$
\theta_k = \frac{\pi}{2} + \frac{(2k-1)\pi}{12}
$$

Evaluando para $k = 1,2,\dots,6$, se obtiene:

$$
\theta_k = 105^\circ,\ 135^\circ,\ 165^\circ,\ 195^\circ,\ 225^\circ,\ 255^\circ
$$

Estos seis polos son los que pertenecen al semiplano izquierdo. De manera equivalente, si los ángulos se miden respecto del eje real negativo, se obtienen:

$$
\psi = \pm 15^\circ,\ \pm 45^\circ,\ \pm 75^\circ
$$

Esta forma es conveniente porque permite agrupar los polos complejos conjugados en tres pares. Cada par genera un factor cuadrático del denominador de la forma:

$$
s^2 + 2\cos(\psi)s + 1
$$

Por lo tanto, para el filtro Butterworth de orden 6 se obtienen tres factores de segundo orden.

Para $\psi = 75^\circ$:

$$
s^2 + 2\cos(75^\circ)s + 1
$$

Como:

$$
2\cos(75^\circ) = 0.5176
$$

queda:

$$
\boxed{s^2 + 0.5176s + 1}
$$

Para $\psi = 45^\circ$:

$$
s^2 + 2\cos(45^\circ)s + 1
$$

Como:

$$
2\cos(45^\circ) = 1.4142
$$

queda:

$$
\boxed{s^2 + 1.4142s + 1}
$$

Para $\psi = 15^\circ$:

$$
s^2 + 2\cos(15^\circ)s + 1
$$

Como:

$$
2\cos(15^\circ) = 1.9319
$$

queda:

$$
\boxed{s^2 + 1.9319s + 1}
$$

De esta manera, el polinomio Butterworth de sexto orden resulta:

$$
B_6(s) =
(s^2 + 0.5176s + 1)
(s^2 + 1.4142s + 1)
(s^2 + 1.9319s + 1)
$$

Como la aproximación Butterworth corresponde a un filtro de tipo `Todo-Polo`, la función transferencia pasa bajos normalizada se expresa como:

$$
T_{LPF}(s) = \frac{1}{B_6(s)}
$$

Por lo tanto:

$$
\boxed{T_{LPF}(s) =
\frac{1}
{(s^2 + 0.5176s + 1)
(s^2 + 1.4142s + 1)
(s^2 + 1.9319s + 1)}}
$$

Finalmente, desarrollando el producto de los factores cuadráticos, se obtiene:

$$
B_6(s) =
s^6 + 3.8637s^5 + 7.4641s^4 + 9.1416s^3 + 7.4641s^2 + 3.8637s + 1
$$

Por lo tanto, la función transferencia pasa bajos Butterworth normalizada de orden 6 queda:

$$
\boxed{T_{LPF}(s) =
\frac{1}
{s^6 + 3.8637s^5 + 7.4641s^4 + 9.1416s^3 + 7.4641s^2 + 3.8637s + 1}}
$$

Esta transferencia corresponde a una aproximación de máxima planicidad, ya que la respuesta Butterworth se caracteriza por no presentar ondulaciones en la banda de paso y por tener una atenuación que aumenta con el orden del filtro.

### <ins>Obtención mediante tabla de coeficientes normalizados

Otra forma más directa de obtener el polinomio Butterworth consiste en utilizar la tabla de coeficientes normalizados de los polinomios de Butterworth.

<div align="center">
    <img src="./Imagenes/Tabla_CoeficientesNormalizadosButterworth.png" alt="Enunciado" width="700"/>
</div>

El denominador normalizado puede expresarse de la forma:

$$
B_n(s) = a_0s^n + a_1s^{n-1} + a_2s^{n-2} + \cdots + a_{n-1}s + a_n
$$

Para los polinomios Butterworth normalizados se cumple que:

$$
a_0 = a_n = 1
$$

Como en este caso el filtro es de orden 6, se busca en la tabla la fila correspondiente a:

$$
n = 6
$$

De la tabla se obtienen los siguientes coeficientes:

$$
a_1 = 3.8637033
$$

$$
a_2 = 7.4641016
$$

$$
a_3 = 9.1416202
$$

$$
a_4 = 7.4641016
$$

$$
a_5 = 3.8637033
$$

Además, como $a_0 = a_6 = 1$, el polinomio Butterworth de sexto orden queda:

$$
B_6(s) =
s^6 + 3.8637033s^5 + 7.4641016s^4 + 9.1416202s^3 + 7.4641016s^2 + 3.8637033s + 1
$$

Finalmente:

$$
\boxed{T_{LPF}(s) =
\frac{1}
{s^6 + 3.8637033s^5 + 7.4641016s^4 + 9.1416202s^3 + 7.4641016s^2 + 3.8637033s + 1}}
$$

Tambien se puede obtener por tabla la forma factorizada

<div align="center">
    <img src="./Imagenes/DenominadorNormalizado_Factorizado.png" alt="Enunciado" width="500"/>
</div>

$$
\boxed{T_{LPF}(s) =
\frac{1}
{(s^2 + 0.5176s + 1)
(s^2 + 1.4142s + 1)
(s^2 + 1.9319s + 1)}}
$$

De esta manera, la tabla permite obtener directamente el mismo resultado que se obtiene mediante la ubicación de los polos, pero de una forma más sencilla y rápida.

---


## 2) Respuesta de módulo, fase y diagrama de polos y ceros

A partir de la función transferencia pasa bajos Butterworth normalizada de orden 6:

$$
T_{LPF}(s) =
\frac{1}
{(s^2 + 0.5176s + 1)
(s^2 + 1.4142s + 1)
(s^2 + 1.9319s + 1)}
$$

se puede analizar cualitativamente su respuesta en frecuencia y su diagrama de polos y ceros.

#### <ins>Respuesta de módulo

Para un filtro Butterworth normalizado de orden $n$, la magnitud al cuadrado está dada por:

$$
|T_n(j\omega)|^2 = \frac{1}{1+\omega^{2n}}
$$

Como en este caso el filtro es de orden 6:

$$
n = 6
$$

entonces:

$$
|T_6(j\omega)|^2 = \frac{1}{1+\omega^{12}}
$$

Por lo tanto, la magnitud resulta:

$$
|T_6(j\omega)| = \frac{1}{\sqrt{1+\omega^{12}}}
$$

Para bajas frecuencias, es decir $\omega \ll 1$, el término $\omega^{12}$ tiende a cero. Por lo tanto:

$$
|T_6(j\omega)| \approx 1
$$

Esto indica que el filtro deja pasar las bajas frecuencias prácticamente sin atenuación.

En la frecuencia normalizada $\omega = 1$:

$$
|T_6(j1)| = \frac{1}{\sqrt{1+1^{12}}}
$$

$$
|T_6(j1)| = \frac{1}{\sqrt{2}} \approx 0.707
$$

Expresado en decibeles:

$$
20\log_{10}(0.707) \approx -3\,\text{dB}
$$

Por lo tanto, $\omega = 1$ corresponde a la frecuencia de corte normalizada del filtro.

Para altas frecuencias, es decir $\omega \gg 1$, la magnitud del filtro Butterworth de orden 6 puede aproximarse como:

$$
|T_6(j\omega)| =
\frac{1}{\sqrt{1+\omega^{12}}}
$$

Como para altas frecuencias se cumple que $\omega^{12} \gg 1$, entonces:

$$
1+\omega^{12} \approx \omega^{12}
$$

Por lo tanto:

$$
|T_6(j\omega)| \approx \frac{1}{\sqrt{\omega^{12}}}
$$

$$
|T_6(j\omega)| \approx \frac{1}{\omega^6}
$$

Expresando esta magnitud en decibeles:

$$
20\log_{10}|T_6(j\omega)|
=
20\log_{10}\left(\frac{1}{\omega^6}\right)
$$

$$
20\log_{10}|T_6(j\omega)|
=
20\log_{10}\left(\omega^{-6}\right)
$$

Aplicando propiedades logarítmicas:

$$
20\log_{10}|T_6(j\omega)|
=
-120\log_{10}(\omega)
$$

Esto significa que, por cada década de aumento en la frecuencia, la magnitud disminuye aproximadamente $120\,\text{dB}$. De forma general, en un filtro pasa bajos de orden $n$, cada polo aporta una pendiente de $-20\,\text{dB/década}$, por lo que la pendiente total es:

$$
-20n\ \text{dB/década}
$$

Para $n = 6$:

$$
-20 \cdot 6 = -120\ \text{dB/década}
$$
Por lo tanto, la respuesta de módulo se mantiene aproximadamente plana en $0\,\text{dB}$ para bajas frecuencias, cae $-3\,\text{dB}$ en $\omega = 1$ y luego decrece con una pendiente aproximada de $-120\,\text{dB/década}$.

<div align="center">
    <img src="./Imagenes/RespuestaModulo.png" alt="Enunciado" width="700"/>
</div>

#### <ins>Respuesta de fase

El filtro Butterworth de orden 6 posee seis polos y ningún cero finito. Cada polo aporta, para altas frecuencias, una variación de fase de aproximadamente $-90^\circ$.

Por lo tanto, la fase total para altas frecuencias tiende a:

$$
\varphi(\infty) = -6 \cdot 90^\circ
$$

$$
\varphi(\infty) = -540^\circ = -3\pi[rad]
$$

Para bajas frecuencias, la fase comienza aproximadamente en:

$$
\varphi(0) \approx 0^\circ
$$

A medida que aumenta la frecuencia, la fase comienza a disminuir. La transición principal ocurre alrededor de la frecuencia de corte normalizada $\omega = 1$. Finalmente, para frecuencias mucho mayores que la frecuencia de corte, la fase tiende asintóticamente a $-540^\circ = -3\pi[rad]$.

<div align="center">
    <img src="./Imagenes/RespuestaFase.png" alt="Enunciado" width="700"/>
</div>

#### <ins>Diagrama de polos y ceros

La aproximación Butterworth corresponde a un filtro de tipo `Todo-Polo`, es decir, un filtro que posee únicamente polos y no posee ceros finitos.

Por lo tanto:

$$
\text{ceros finitos} = 0
$$

Los polos de un Butterworth normalizado de orden 6 se ubican sobre una circunferencia de radio unitario en el semiplano izquierdo del plano $s$. Sus ángulos absolutos, medidos desde el eje real positivo, son:

$$
105^\circ,\ 135^\circ,\ 165^\circ,\ 195^\circ,\ 225^\circ,\ 255^\circ
$$

Partiendo de la transferencia factorizada se pueden despejar los polos facilmente:
$$
\boxed{T_{LPF}(s) =
\frac{1}
{(s^2 + 0.5176s + 1)
(s^2 + 1.4142s + 1)
(s^2 + 1.9319s + 1)}}
$$

$$
p_1 = -0.2588 + j0.9659
$$

$$
p_2 = -0.7071 + j0.7071
$$

$$
p_3 = -0.9659 + j0.2588
$$

$$
p_4 = -0.9659 - j0.2588
$$

$$
p_5 = -0.7071 - j0.7071
$$

$$
p_6 = -0.2588 - j0.9659
$$

Todos los polos tienen parte real negativa, por lo que el filtro resultante es estable.

El diagrama de polos y ceros presenta seis polos distribuidos simétricamente sobre una semicircunferencia de radio unitario en el semiplano izquierdo, y no presenta ceros finitos.

<div align="center">
    <img src="./Imagenes/Polos.png" alt="Enunciado" width="500"/>
</div>